In [ ]:
# Phase (I): Test GRPO on Game of 24 — surface drawbacks D1-D4
#
#   D1  CoT redundancy           → track CoT length distribution across training
#   D2  No per-token signal      → (structural; visualised by absence of v_t)
#   D3  Zero-pass@K dead zone    → curate hard puzzles, track which never solve
#   D4  Indiscriminate credit    → inspect failed rollouts that share good prefixes
#
# Phase (II) [later]: swap reward + advantage for velocity / answer-buffer / prefix-buffer
#                     and re-run the same diagnostics to verify the four fixes.

In [ ]:
import os, re, json, random, itertools, math
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
random.seed(0); np.random.seed(0); torch.manual_seed(0)

MODEL_NAME    = "Qwen/Qwen3-0.6B"
OUTPUT_DIR    = Path("output/game24_grpo_baseline")
ROLLOUT_LOG   = OUTPUT_DIR / "rollouts.jsonl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR.resolve())

## 2. Game-of-24 &nbsp;·&nbsp; data &nbsp;·&nbsp; rewards

Everything in this section is a thin wrapper around `src.game24utils`:

- **verifier + solver** — `verify_24`, `enumerate_solutions`
- **puzzle pool + buckets** — `build_puzzle_pool`, `bucket_by_difficulty`
- **splits** — `make_splits` (proportion-based; holds out hard puzzles for D3)
- **prompts / datasets** — `SYSTEM_PROMPT`, `to_chat`, `build_datasets`
- **rewards** — `correctness_reward`, `format_reward`

The notebook itself focuses on training, diagnostics, and the velocity-reward
analysis. Algorithm pieces stay here; bookkeeping pieces live in the module.

In [ ]:
from src.game24utils import (
    TARGET, verify_24, enumerate_solutions,
    build_puzzle_pool, bucket_by_difficulty, make_splits,
    SYSTEM_PROMPT, to_chat, build_datasets,
    completion_text, extract_expr, correctness_reward, format_reward,
)

# All solvable 4-tuples from digits 1..9, bucketed by # solutions.
puzzles = build_puzzle_pool(max_n=9)
easy, medium, hard = bucket_by_difficulty(puzzles, easy_min=8, hard_max=2)

print(f"Solvable 4-tuples: {len(puzzles)}")
print(f"  easy   (≥8 sols): {len(easy)}")
print(f"  medium (3-7 sols): {len(medium)}")
print(f"  hard   (≤2 sols): {len(hard)}")
print("example hard puzzle:", hard[0] if hard else None)

In [ ]:
# Proportion-based splits.
# - eval_frac: held-out slice of easy+medium for in-distribution eval
# - probe_frac: slice of HARD puzzles held out as the D3 zero-pass@K probe
train_puzzles, eval_puzzles, hard_probe = make_splits(
    easy, medium, hard, eval_frac=0.10, probe_frac=0.40,
)

print(f"train={len(train_puzzles)}, eval={len(eval_puzzles)}, "
      f"probe(hard, held-out)={len(hard_probe)}")

train_ds, eval_ds, probe_ds = build_datasets(train_puzzles, eval_puzzles, hard_probe)
print(train_ds[0])

In [ ]:
# Rewards live in src.game24utils — trajectory-level only (D2 is structural).
# Sanity:
fake = [[{"role": "assistant", "content": "Let me think...\n#### (3+5)*(7-4)"}]]
print("correctness:", correctness_reward(fake, numbers=[[3, 5, 7, 4]]))
print("format    :", format_reward(fake))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RolloutLogger:
    """Logs every rollout to JSONL via the reward-fn callback. Returns 0.0 reward."""
    def __init__(self, path: Path, tok):
        self.path = path
        self.tok  = tok
        self.step = 0
        self.path.write_text("")  # truncate

    def __call__(self, completions, numbers, solutions=None, **kwargs):
        with self.path.open("a") as f:
            for i, (c, nums) in enumerate(zip(completions, numbers)):
                text = _text(c)
                expr = extract_expr(text)
                correct = verify_24(list(nums), expr)
                n_tok = len(self.tok.encode(text, add_special_tokens=False))
                f.write(json.dumps({
                    "step": self.step,
                    "idx": i,
                    "numbers": list(nums),
                    "completion": text,
                    "expr": expr,
                    "correct": bool(correct),
                    "n_tokens": int(n_tok),
                }) + "\n")
        self.step += 1
        return [0.0] * len(completions)   # passthrough, contributes nothing

    # Keep TRL happy: it inspects __name__ on reward callables.
    __name__ = "rollout_logger"

rollout_logger = RolloutLogger(ROLLOUT_LOG, tokenizer)
print("Rollout log →", ROLLOUT_LOG)

## 5. Train &nbsp;·&nbsp; vanilla GRPO

Short run (200 steps, 8 generations/prompt) is enough to surface the four
drawbacks. Bump `max_steps` for a longer collapse-watch.

In [ ]:
config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_generations=8,
    max_completion_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=200,
    logging_steps=5,
    bf16=True,
    save_strategy="no",
    report_to="none",
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.4,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward, rollout_logger],
    args=config,
    train_dataset=train_ds,
)
trainer.train()
print("Done. Rollouts at:", ROLLOUT_LOG)

## 6. Diagnostics — surfacing D1, D3, D4

We load the rollout log and compute four lenses. D2 is structural (there is no
per-token signal to plot) — its absence is the diagnostic.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rollouts = [json.loads(l) for l in ROLLOUT_LOG.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rollouts)
df["key"] = df["numbers"].apply(lambda x: tuple(sorted(x)))
print(f"{len(df)} rollouts logged across {df['step'].nunique()} reward-fn calls")
df.head(3)

In [ ]:
# ── D1: CoT redundancy — length + pairwise edit-distance of correct rollouts ──
# Length is the obvious probe. A cleaner second signal: among *correct*
# rollouts on the same puzzle, how diverse are the CoT strings?
# Collapse to a single solution form shows up as edit-distance → 0.

agg = df.groupby("step").agg(
    n_tokens_mean=("n_tokens", "mean"),
    n_tokens_p90=("n_tokens", lambda s: np.percentile(s, 90)),
    acc=("correct", "mean"),
).reset_index()

def norm_edit(a: str, b: str) -> float:
    # Cheap token-level proxy: 1 - LCS/max_len, via difflib.
    import difflib
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    return 1.0 - sm.ratio()

ed_rows = []
for (step, key), g in df[df.correct].groupby(["step", "key"]):
    texts = g.completion.tolist()
    if len(texts) < 2: continue
    pairs = [norm_edit(texts[i], texts[j])
             for i in range(len(texts)) for j in range(i+1, len(texts))]
    ed_rows.append({"step": step, "mean_edit": np.mean(pairs), "n_pairs": len(pairs)})
edf = pd.DataFrame(ed_rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(agg.step, agg.n_tokens_mean, label="mean")
axes[0].plot(agg.step, agg.n_tokens_p90,  label="p90", linestyle="--")
axes[0].set_xlabel("step"); axes[0].set_ylabel("CoT tokens")
axes[0].set_title("D1 · CoT length"); axes[0].legend()

for ok, label in [(True, "correct"), (False, "incorrect")]:
    sub = df[df.correct == ok].groupby("step").n_tokens.mean()
    axes[1].plot(sub.index, sub.values, label=label)
axes[1].set_xlabel("step"); axes[1].set_ylabel("mean tokens")
axes[1].set_title("D1 · length, split by correctness"); axes[1].legend()

if len(edf):
    axes[2].plot(edf.step, edf.mean_edit)
    axes[2].set_xlabel("step"); axes[2].set_ylabel("mean pairwise edit-distance")
    axes[2].set_title("D1 · diversity of correct CoTs\n(collapse → 0)")
else:
    axes[2].text(0.5, 0.5, "not enough correct rollouts\non same puzzle to measure",
                 ha="center", va="center"); axes[2].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── Diversity probe — motivates the answer buffer A_q (NOT D2) ──
# For each puzzle with ≥1 solution found, how many *distinct* correct
# expressions did the model produce over the whole run? Compared to the
# total number of solutions the puzzle admits.
def canon(expr: str) -> str:
    """Normalise whitespace so "1 + 2" and "1+2" collapse to one form."""
    return re.sub(r"\s+", "", expr)

rows = []
for key, g in df[df.correct].groupby("key"):
    puzzle = next((p for p in train_puzzles if tuple(sorted(p["numbers"])) == key), None)
    if puzzle is None: continue
    unique = len(set(canon(e) for e in g.expr))
    rows.append({"key": key, "unique_found": unique,
                 "total_solutions": puzzle["n_solutions"],
                 "coverage": unique / puzzle["n_solutions"]})
ddf = pd.DataFrame(rows)
if len(ddf):
    print(f"Puzzles with ≥1 solution found: {len(ddf)}")
    print(f"  mean unique solutions found per puzzle: {ddf.unique_found.mean():.2f}")
    print(f"  mean total solutions per puzzle:        {ddf.total_solutions.mean():.2f}")
    print(f"  mean coverage:                          {ddf.coverage.mean():.1%}")
    plt.figure(figsize=(6, 3))
    plt.hist(ddf.coverage, bins=20)
    plt.xlabel("fraction of a puzzle's solutions the model produced")
    plt.ylabel("# puzzles")
    plt.title("Diversity probe — motivates answer buffer A_q\n"
              "(low coverage → model collapses to few solution forms)")
    plt.tight_layout(); plt.show()

In [ ]:
# ── D3: zero-pass@K dead-zone — which puzzles never get solved? ──
solved_per_puzzle = df.groupby("key").correct.any()
n_solved   = int(solved_per_puzzle.sum())
n_attempted = int(len(solved_per_puzzle))
print(f"Puzzles ever solved during training: {n_solved}/{n_attempted}  "
      f"({n_solved/n_attempted:.1%})")

# Cross-reference with the # of solutions per puzzle
puzzle_info = pd.DataFrame([
    {"key": tuple(sorted(p["numbers"])), "n_solutions": p["n_solutions"]}
    for p in train_puzzles
]).drop_duplicates("key").set_index("key")
joined = puzzle_info.join(solved_per_puzzle.rename("ever_solved"))
bucketed = joined.groupby(pd.cut(joined.n_solutions, [0, 2, 7, 100],
                                  labels=["hard (≤2)", "med (3-7)", "easy (≥8)"])).ever_solved.agg(["mean", "count"])
print("\nPass-rate by difficulty bucket:")
print(bucketed)
print("\nD3 signature: hard-bucket pass-rate stays at ~0 → these puzzles "
      "receive no learning signal because GRPO has no positive trajectory "
      "in their group.")

In [ ]:
# ── v_t helper — post-hoc decoding-velocity, used by D2 and D4 below ──
# v_t = log p(ref_answer | q, o_{1:t}) - log p(ref_answer | q, o_{1:t-1})
# Slow (T+1 forward passes per rollout); applied only to a subsampled set.

device = "cuda" if torch.cuda.is_available() else "cpu"
scorer_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device).eval()

def _split_completion(completion: str):
    m = re.search(r"####\s*(.*)$", completion.strip(), re.DOTALL)
    if not m:
        return completion, ""
    return completion[: m.start()].rstrip(), f"#### {m.group(1).strip()}"

def _prompt_text(prompt_messages) -> str:
    return tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )

def _lookup_puzzle(key):
    for p in train_puzzles + eval_puzzles + hard_probe:
        if tuple(sorted(p["numbers"])) == key:
            return p
    return None

@torch.no_grad()
def compute_vt(prompt_messages, completion_text, reference_answer,
               tok=tokenizer, model=scorer_model):
    q_text = _prompt_text(prompt_messages)
    cot, _ = _split_completion(completion_text)
    a_text = reference_answer if reference_answer.startswith("####") else f"#### {reference_answer}"

    q_ids = tok(q_text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    o_ids = tok(cot,    return_tensors="pt", add_special_tokens=False).input_ids[0]
    a_ids = tok(a_text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    T, La = len(o_ids), len(a_ids)

    logps = []
    for t in range(T + 1):
        ids = torch.cat([q_ids, o_ids[:t], a_ids]).unsqueeze(0).to(device)
        logits = model(ids).logits[0]
        start = len(q_ids) + t
        lp = sum(
            torch.log_softmax(logits[start + i - 1], dim=-1)[a_ids[i]].item()
            for i in range(La)
        )
        logps.append(lp)

    vt = [logps[t] - logps[t - 1] for t in range(1, T + 1)]
    tok_strs = tok.convert_ids_to_tokens(o_ids)
    return tok_strs, vt, logps
print("v_t scorer ready on", device)

In [ ]:
# ── D2: post-hoc v_t overlay — the signal GRPO threw away ──
# We compute v_t for one correct and one incorrect rollout and plot it
# alongside GRPO's broadcast advantage. The visual contrast (flat line vs.
# rich per-token variation) IS the D2 diagnostic.

def _plot_vt(ax, toks, vt, title, broadcast_adv):
    ax.bar(range(len(vt)), vt, color=["#2a9d8f" if v >= 0 else "#e76f51" for v in vt])
    ax.axhline(broadcast_adv, color="k", linestyle="--", alpha=0.7,
               label=f"GRPO broadcast adv = {broadcast_adv:+.2f}")
    ax.set_title(title); ax.set_xlabel("token position t"); ax.set_ylabel("v_t")
    ax.legend(loc="upper right", fontsize=8)
    # Sparse token labels to avoid clutter
    step = max(1, len(toks) // 20)
    ax.set_xticks(range(0, len(toks), step))
    ax.set_xticklabels([toks[i].replace("Ġ", "▁")[:6] for i in range(0, len(toks), step)],
                       rotation=45, fontsize=7)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, correct_flag, title in [(axes[0], True,  "correct rollout  (GRPO adv ≈ +A)"),
                                (axes[1], False, "incorrect rollout (GRPO adv ≈ −A)")]:
    sub = df[df.correct == correct_flag]
    if len(sub) == 0:
        ax.text(0.5, 0.5, f"no {'correct' if correct_flag else 'incorrect'} rollouts",
                ha="center", va="center"); ax.axis("off"); continue
    row = sub.sample(1, random_state=1).iloc[0]
    puzzle = _lookup_puzzle(row.key)
    ref_answer = puzzle["solutions"][0] if puzzle else row.expr
    toks, vt, _ = compute_vt(to_chat(puzzle)["prompt"], row.completion, ref_answer)
    _plot_vt(ax, toks, vt, f"{title}  ·  puzzle={row.numbers}", +1.0 if correct_flag else -1.0)

plt.suptitle("D2 · per-token velocity vs. GRPO's single broadcast advantage", y=1.02)
plt.tight_layout(); plt.show()

print("D2 signature: v_t varies sharply per token, but GRPO's dashed line is flat.")
print("  Phase II replaces that dashed line with the coloured bars.")

In [ ]:
# ── D4: indiscriminate credit — v_t-based probe on FAILED rollouts ──
# Claim: many tokens inside failed rollouts have v_t > 0 (locally moved
# decoding toward the *correct* answer), yet GRPO's trajectory-level
# negative advantage penalises them uniformly.

FAIL_SAMPLE_N = 8
fail_pool = df[~df.correct].sample(min(FAIL_SAMPLE_N, (~df.correct).sum()),
                                   random_state=0)

d4_rows = []
for _, row in fail_pool.iterrows():
    puzzle = _lookup_puzzle(row.key)
    if puzzle is None: continue
    ref_answer = puzzle["solutions"][0]
    toks, vt, _ = compute_vt(to_chat(puzzle)["prompt"], row.completion, ref_answer)
    pos = sum(1 for v in vt if v > 0)
    d4_rows.append({"key": row.key, "n_tokens": len(vt),
                    "n_vt_positive": pos,
                    "frac_positive": pos / max(1, len(vt)),
                    "total_vt": sum(vt)})
d4df = pd.DataFrame(d4_rows)

print(f"Probed {len(d4df)} failed rollouts")
if len(d4df):
    print(f"  mean fraction of tokens with v_t > 0: {d4df.frac_positive.mean():.1%}")
    print(f"  → these tokens LOCALLY improved decoding of the reference answer,")
    print(f"    yet GRPO assigns them the same negative advantage as every other token.")
    plt.figure(figsize=(6, 3))
    plt.hist(d4df.frac_positive, bins=20)
    plt.xlabel("fraction of tokens with v_t > 0 (failed rollouts)")
    plt.ylabel("# rollouts")
    plt.title("D4 · productive tokens inside failed trajectories\n"
              "GRPO penalises them indiscriminately")
    plt.tight_layout(); plt.show()

## 7. Summary &nbsp;·&nbsp; probe → drawback mapping

| Drawback | Probe | Expected signature |
|---|---|---|
| **D1** | CoT length over steps · pairwise edit-distance of correct rollouts | length grows or stays high; edit-distance of correct CoTs → 0 (mode collapse) |
| **D2** | post-hoc $v_t$ overlay on one correct + one incorrect rollout | $v_t$ varies sharply token-by-token, but GRPO's broadcast advantage is a flat dashed line |
| **D3** | pass-rate bucketed by `n_solutions`; hard-bucket puzzles | hard-bucket (≤2 sols) pass-rate stays ≈ 0 throughout training |
| **D4** | fraction of tokens with $v_t > 0$ inside *failed* rollouts | non-trivial fraction of productive tokens, uniformly penalised by GRPO |
| *Diversity (→ motivates $\mathcal{A}_q$)* | unique solutions found per puzzle / total | low coverage → model collapses to few solution forms |

Phase (II) swaps reward + advantage for velocity / answer-buffer /
prefix-buffer and re-runs these same probes. Each row should flip.